In [1]:
import pandas as pd
import numpy as np

TARGETS = ["y"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_y,MSE_ZZx1_y,R2_ZZx2_y,MSE_ZZx2_y,R2_ZZxReto_y,MSE_ZZxReto_y
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed4080,[1],0.5,0.5,0.01,4080,0.973120,0.922055,0.970488,0.843703,-4.909024,0.076342
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed1926,[1],0.5,0.5,0.01,1926,0.972931,0.923526,0.971375,0.849837,-4.813950,0.082241
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed679,[1],0.5,0.5,0.01,679,0.973275,0.922303,0.971056,0.845237,-4.890363,0.077699
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed5929,[1],0.5,0.5,0.01,5929,0.973075,0.923449,0.971353,0.849124,-4.834555,0.081021
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed896,[1],0.5,0.5,0.01,896,0.973277,0.922309,0.970911,0.844985,-4.882662,0.077867
...,...,...,...,...,...,...,...,...,...,...,...,...
2422,model_arch81_r0.9_Ld0.5_Lp0.5_seed6833,[81],0.5,0.5,0.90,6833,0.980676,0.941323,0.967013,0.874201,-3.197185,0.274780
2423,model_arch81_r0.9_Ld0.5_Lp0.5_seed3174,[81],0.5,0.5,0.90,3174,0.976415,0.938399,0.971854,0.878716,-3.695895,0.235064
2424,model_arch81_r0.9_Ld0.5_Lp0.5_seed7907,[81],0.5,0.5,0.90,7907,0.981987,0.945161,0.945883,0.877721,-2.832436,0.321128
2425,model_arch81_r0.9_Ld0.5_Lp0.5_seed6039,[81],0.5,0.5,0.90,6039,0.983731,0.948146,0.943646,0.884820,-2.587495,0.345267


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - y


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2250,model_arch76_r0.01_Ld0.5_Lp0.5_seed4080,[76],0.994326,0.986325,0.818080,0.913629
1824,model_arch61_r0.01_Ld0.7_Lp0.3_seed896,[61],0.999734,0.968144,0.782156,0.895755
1646,model_arch55_r0.9_Ld0.7_Lp0.3_seed1926,[55],0.999716,0.992967,0.756844,0.893513
1982,model_arch67_r0.01_Ld0.5_Lp0.5_seed679,[67],0.998719,0.982683,0.640544,0.845010
1809,model_arch61_r0.9_Ld0.5_Lp0.5_seed896,[61],0.999376,0.978985,0.635181,0.842005



📊 MÉTRICAS COMPLETAS - TOP 5 (y)


,model,Neurons,R2_ZZx1_y,R2_ZZx2_y,R2_ZZxReto_y,R2_train_mean,R2_val_mean,R2_test_mean,Score
2250,model_arch76_r0.01_Ld0.5_Lp0.5_seed4080,[76],0.994326,0.986325,0.818080,0.994326,0.986325,0.818080,0.913629
1824,model_arch61_r0.01_Ld0.7_Lp0.3_seed896,[61],0.999734,0.968144,0.782156,0.999734,0.968144,0.782156,0.895755
1646,model_arch55_r0.9_Ld0.7_Lp0.3_seed1926,[55],0.999716,0.992967,0.756844,0.999716,0.992967,0.756844,0.893513
1982,model_arch67_r0.01_Ld0.5_Lp0.5_seed679,[67],0.998719,0.982683,0.640544,0.998719,0.982683,0.640544,0.845010
1809,model_arch61_r0.9_Ld0.5_Lp0.5_seed896,[61],0.999376,0.978985,0.635181,0.999376,0.978985,0.635181,0.842005


In [5]:
final_table.to_excel("BestModels-otm.xlsx")